# 04 - Tier 2: the agent that uses tools

Tier 1 answered from memory and made numbers up. Tier 2 can actually run
queries on our database.

How it works:

1. **Plan** - the model sees the schema and decides what queries to run
2. **Run** - we execute those queries and save everything in a log
3. **Answer** - the model sees the real rows and writes the answer

We use two model calls instead of one. If we asked for the answer in one go,
the model would have to guess what its own query returns, which is exactly
how made-up numbers get in.

The code lives in `src/evidenceiq.py` so all the notebooks can share it.


## Setup


In [1]:
import sys
sys.path.append('../src')

import evidenceiq as eiq


## 1. The tools, one at a time

Before letting the model use them, let's check they work.


### run_sql


In [2]:
log = []      # the log is just a list of dictionaries

eiq.run_sql("SELECT ROUND(SUM(revenue),2) AS revenue FROM sales "
            "WHERE NOT is_cancellation AND year(invoice_date)=2011", log)

log[0]


{'n': 0,
 'tool': 'run_sql',
 'code': 'SELECT ROUND(SUM(revenue),2) AS revenue FROM sales WHERE NOT is_cancellation AND year(invoice_date)=2011',
 'ok': True,
 'rows': [{'revenue': 9809614.01}],
 'error': None}

That should be **9,809,614.01** - the same number as q01 in our benchmark.


### The database is read-only

The agent writes its own SQL, so we have to stop it changing the data.
We check the query first, and we also open DuckDB with `read_only=True`.


In [3]:
bad = eiq.run_sql('DROP TABLE sales', log)
print('ok?    ', bad['ok'])
print('error: ', bad['error'])


ok?     False
error:  not a read-only query


### run_python

For things that are awkward in SQL, like a percentage change.


In [4]:
eiq.run_python("result = {'pct': round((1456145.80-1069368.23)/1069368.23*100, 2)}", log)
log[-1]['rows']


[{'pct': 36.17}]

### The log

Everything the agent does ends up here. This is the important bit - the
verifier in notebook 05 will only trust numbers that appear in this log.


In [5]:
for call in log:
    print(call['n'], call['tool'], 'ok' if call['ok'] else 'FAILED')

print()
print('every number the tools returned:')
print(eiq.numbers_in_log(log))


0 run_sql ok
1 run_sql FAILED
2 run_python ok

every number the tools returned:
[(0, 9809614.01), (2, 36.17)]


## 2. The prompt

The rules matter less than the examples. Gemma will not remember
"exclude cancellations" from a sentence, but it will copy a query it has
just been shown. So every rule from `docs/kpi_definitions.md` is in there
as a worked example.


In [6]:
print(eiq.PLAN_PROMPT[:1500])


You are a business data analyst. Plan the queries needed to answer
the question. Do NOT answer yet - you have not seen any data.

Database: a UK online gift wholesaler, Dec 2009 to 9 Dec 2011, 1,033,030 rows.

TABLE sales (one row per invoice line)
  invoice_no, stock_code, description, quantity, unit_price, customer_id,
  country, revenue (= quantity * unit_price), invoice_date, invoice_month,
  is_cancellation, is_product, is_outlier

TABLE dim_month   invoice_month, trading_days, net_revenue, is_complete_month
TABLE dim_product stock_code, description, units_sold, gross_revenue
TABLE dim_customer customer_id, country, first_order, last_order, orders, net_revenue

There is NO cost, profit, margin, discount, competitor or customer age data.

RULES (the answer is wrong without these):
1. Revenue always excludes cancellations:  WHERE NOT is_cancellation
2. Product questions also need:            AND is_product AND NOT is_outlier
3. Group products by stock_code, and use mode(description)

## 3. Run Tier 2 on one question

Make sure Ollama is running first: `ollama pull gemma3`


In [7]:
answer = eiq.tier2('What was our total revenue in 2011?')

print('PLAN:')
for step in answer['plan']:
    print(' -', step)

print()
print('FINDINGS:', answer['findings'])


ResponseError: model 'gemma3' not found (status code: 404)

### What SQL did it write?


In [ ]:
for call in answer['log']:
    print('[%d] %s   ok=%s' % (call['n'], call['tool'], call['ok']))
    print(call['code'])
    print('->', call['rows'] if call['ok'] else call['error'])
    print()


### The claims

The model has to list every number separately, not just write a paragraph.
Notebook 05 needs this - you cannot reliably pull numbers out of prose.


In [ ]:
for c in answer['claims']:
    print(c['value'], c['unit'], '|', c['text'])
    print('    from call', c.get('from_call'), '| calc:', c.get('calc'))


## 4. Does it refuse impossible questions?

There is no cost data, so profit margin cannot be worked out.
A good answer says so. A bad one invents a number.


In [ ]:
bad_q = eiq.tier2('What was our profit margin in 2011?')

print('refused?    ', bad_q['insufficient_data'])
print('queries run:', len(bad_q['log']))
print('says:       ', bad_q['findings'])


## 5. Try the harder ones

q11 and q12 from the benchmark. q12 is the hardest - March had 27 trading
days and April had 21, so the honest answer gives the per-day figure too.


In [ ]:
for q in ['Did November 2011 beat October 2011 on revenue, and by how much?',
          'Which five products generated the most revenue in 2011?']:
    print('=' * 70)
    print(q)
    print('=' * 70)
    a = eiq.tier2(q)
    print(a['findings'])
    print()


## What we found

Write down what went wrong here - it goes in the report:

- did it remember `NOT is_cancellation`?
- did any query fail, and why?
- did it fill in `calc` and `inputs` for calculated numbers?

**When Gemma gets a query wrong, fix the prompt, not the code.** Add another
worked example to `EXAMPLES` in `src/evidenceiq.py`.

Tier 2 numbers are now real. But nothing checks that the *sentence* matches
the rows - that is notebook 05.
